# Pronóstico de Volumen y Planeación de Dotación (Workforce Management)

**Autor:** Juan Diego Roncancio Melo

**Objetivo:** demostrar un flujo end-to-end de forecasting de demanda aplicado a
planeación de personal (Workforce Management): desde una serie de tiempo de
volumen de contactos hasta la dotación (FTE) requerida por día, con capacidad de
simular escenarios de negocio.

> **Nota sobre los datos:** este proyecto usa una serie de tiempo **sintética**
> (simulada), construida para comportarse como datos reales de un centro de
> servicio (tendencia, estacionalidad semanal y anual, feriados, ruido). No son
> datos reales de ninguna empresa — se simulan para poder publicar el proyecto
> abiertamente. La metodología y las técnicas (SARIMAX, Erlang C) son las mismas
> que se aplicarían sobre datos reales de operación.

**Flujo del proyecto:**
1. Generación de datos sintéticos
2. Análisis exploratorio de la serie de tiempo
3. Modelo de pronóstico (SARIMAX + términos de Fourier)
4. Traducción del pronóstico a dotación (FTE) con el modelo Erlang C
5. App interactiva de escenarios (Streamlit) — código en `05_app_streamlit.py`


## 1. Generación de datos sintéticos

In [1]:
"""
01_generar_datos_sinteticos.py

IMPORTANTE - TRANSPARENCIA SOBRE LOS DATOS:
Este proyecto usa una serie de tiempo SINTETICA (simulada), generada con este
script, NO datos reales de ninguna empresa. Se construye para que se comporte
como una serie real de volumen de contactos de un centro de servicio (tendencia,
estacionalidad semanal, estacionalidad anual, feriados y ruido aleatorio), con el
fin de demostrar de forma honesta el flujo completo de forecasting y planeacion
de dotacion (Workforce Management) sobre una base reproducible y compartible
publicamente (datos reales de una operacion nunca podrian publicarse en GitHub).

Genera:
    data/volumen_contactos.csv  -> serie diaria de volumen de contactos (2023-2025)
"""

import numpy as np
import pandas as pd
from pathlib import Path

OUT_DIR = Path.cwd() / "data"
OUT_DIR.mkdir(exist_ok=True)

np.random.seed(42)

# --- Rango de fechas: 3 años de historico diario ---
start = "2023-01-01"
end = "2025-12-31"
dates = pd.date_range(start=start, end=end, freq="D")
n = len(dates)
t = np.arange(n)

# --- Componente de tendencia: crecimiento gradual de la operacion ---
tendencia = 1400 + 0.35 * t

# --- Estacionalidad semanal: lunes y martes son los dias de mayor volumen,
#     domingo el de menor (patron tipico de servicio al cliente) ---
dow_factor = {
    0: 1.18,  # lunes
    1: 1.10,  # martes
    2: 1.00,  # miercoles
    3: 0.97,  # jueves
    4: 0.95,  # viernes
    5: 0.70,  # sabado
    6: 0.55,  # domingo
}
estacionalidad_semanal = np.array([dow_factor[d.weekday()] for d in dates])

# --- Estacionalidad anual: pico en enero (renovaciones/inicio de ano) y
#     noviembre-diciembre (temporada alta), valle a mitad de ano ---
dia_del_anio = np.array([d.dayofyear for d in dates])
estacionalidad_anual = 1 + 0.18 * np.sin(2 * np.pi * (dia_del_anio - 15) / 365.25) \
                          + 0.10 * np.sin(4 * np.pi * (dia_del_anio - 320) / 365.25)

# --- Feriados colombianos aproximados (reduccion de volumen operativo) ---
feriados = pd.to_datetime([
    "2023-01-01", "2023-01-09", "2023-03-20", "2023-04-06", "2023-04-07",
    "2023-05-01", "2023-05-22", "2023-06-12", "2023-06-19", "2023-07-03",
    "2023-07-20", "2023-08-07", "2023-08-21", "2023-10-16", "2023-11-06",
    "2023-11-13", "2023-12-08", "2023-12-25",
    "2024-01-01", "2024-01-08", "2024-03-25", "2024-03-28", "2024-03-29",
    "2024-05-01", "2024-05-13", "2024-06-03", "2024-06-10", "2024-07-01",
    "2024-07-20", "2024-08-07", "2024-08-19", "2024-10-14", "2024-11-04",
    "2024-11-11", "2024-12-08", "2024-12-25",
    "2025-01-01", "2025-01-06", "2025-03-24", "2025-04-17", "2025-04-18",
    "2025-05-01", "2025-06-02", "2025-06-23", "2025-06-30", "2025-07-20",
    "2025-08-07", "2025-08-18", "2025-10-13", "2025-11-03", "2025-11-17",
    "2025-12-08", "2025-12-25",
])
factor_feriado = np.array([0.45 if d in feriados else 1.0 for d in dates])

# --- Ruido aleatorio (variabilidad diaria propia de cualquier operacion) ---
ruido = np.random.normal(loc=1.0, scale=0.05, size=n)

# --- Serie final ---
volumen = tendencia * estacionalidad_semanal * estacionalidad_anual * factor_feriado * ruido
volumen = np.round(np.clip(volumen, 200, None)).astype(int)

df = pd.DataFrame({"fecha": dates, "volumen_contactos": volumen})
df.to_csv(OUT_DIR / "volumen_contactos.csv", index=False)

print(f"Serie generada: {n} dias, desde {start} hasta {end}")
print(df.describe())
print(f"Guardado en: {OUT_DIR / 'volumen_contactos.csv'}")


Serie generada: 1096 dias, desde 2023-01-01 hasta 2025-12-31
                     fecha  volumen_contactos
count                 1096        1096.000000
mean   2024-07-01 12:00:00        1419.171533
min    2023-01-01 00:00:00         375.000000
25%    2023-10-01 18:00:00        1093.750000
50%    2024-07-01 12:00:00        1444.000000
75%    2025-04-01 06:00:00        1747.000000
max    2025-12-31 00:00:00        2562.000000
std                    NaN         416.629388
Guardado en: /tmp/claude-0/-home-claude/6c8bc701-6644-54f8-94af-291fdfbab41c/scratchpad/forecasting_project/data/volumen_contactos.csv


## 2. Análisis exploratorio de la serie de tiempo

Se evalúa la tendencia, estacionalidad y estacionariedad de la serie antes de modelar.

In [2]:
"""
02_eda_series_tiempo.py

Analisis exploratorio de la serie de tiempo de volumen de contactos:
- Visualizacion general (tendencia + estacionalidad a simple vista)
- Descomposicion estacional (tendencia / estacionalidad / residuo)
- Prueba de estacionariedad (Dickey-Fuller aumentada)
- Autocorrelacion (ACF/PACF) para orientar el orden del modelo SARIMA

Genera figuras en outputs/ para el README y el notebook.
"""

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

BASE = Path.cwd()
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(BASE / "data" / "volumen_contactos.csv", parse_dates=["fecha"])
df = df.set_index("fecha")
serie = df["volumen_contactos"]

# --- 1. Serie completa ---
fig, ax = plt.subplots(figsize=(12, 4))
serie.plot(ax=ax, color="#1F3864", linewidth=0.8)
ax.set_title("Volumen diario de contactos (2023-2025)")
ax.set_ylabel("Contactos/dia")
plt.tight_layout()
plt.savefig(OUT / "01_serie_completa.png", dpi=130)
plt.close()

# --- 2. Zoom a 3 meses para ver estacionalidad semanal ---
fig, ax = plt.subplots(figsize=(12, 4))
serie.loc["2025-01-01":"2025-03-31"].plot(ax=ax, color="#2E75B6", marker="o", markersize=2)
ax.set_title("Zoom Q1 2025 - Patron semanal visible")
ax.set_ylabel("Contactos/dia")
plt.tight_layout()
plt.savefig(OUT / "02_zoom_trimestre.png", dpi=130)
plt.close()

# --- 3. Descomposicion estacional (periodo semanal = 7) ---
decomposicion = seasonal_decompose(serie, model="multiplicative", period=7)
fig = decomposicion.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.savefig(OUT / "03_descomposicion.png", dpi=130)
plt.close()

# --- 4. Prueba de estacionariedad (ADF) sobre la serie y su diferencia semanal ---
adf_original = adfuller(serie.dropna())
serie_diff7 = serie.diff(7).dropna()
adf_diff = adfuller(serie_diff7)

print("=== Prueba Dickey-Fuller Aumentada (ADF) ===")
print(f"Serie original     -> estadistico={adf_original[0]:.3f}, p-valor={adf_original[1]:.4f}")
print(f"Diferenciada (d=7)  -> estadistico={adf_diff[0]:.3f}, p-valor={adf_diff[1]:.4f}")
print()
if adf_original[1] > 0.05:
    print("La serie original NO es estacionaria (p > 0.05) -> requiere diferenciacion estacional.")
else:
    print("La serie original es estacionaria (p <= 0.05).")
if adf_diff[1] <= 0.05:
    print("Tras diferenciar con periodo 7 (semanal), la serie SI es estacionaria.")

# --- 5. ACF / PACF sobre la serie diferenciada para orientar el orden SARIMA ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(serie_diff7, lags=30, ax=axes[0])
axes[0].set_title("ACF (serie diferenciada, d=7)")
plot_pacf(serie_diff7, lags=30, ax=axes[1], method="ywm")
axes[1].set_title("PACF (serie diferenciada, d=7)")
plt.tight_layout()
plt.savefig(OUT / "04_acf_pacf.png", dpi=130)
plt.close()

print(f"\nFiguras guardadas en: {OUT}")


/tmp/ipykernel_835/4115377660.py:57: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_original = adfuller(serie.dropna())
/tmp/ipykernel_835/4115377660.py:59: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_diff = adfuller(serie_diff7)


=== Prueba Dickey-Fuller Aumentada (ADF) ===
Serie original     -> estadistico=-1.534, p-valor=0.5165
Diferenciada (d=7)  -> estadistico=-7.845, p-valor=0.0000

La serie original NO es estacionaria (p > 0.05) -> requiere diferenciacion estacional.
Tras diferenciar con periodo 7 (semanal), la serie SI es estacionaria.



Figuras guardadas en: /tmp/claude-0/-home-claude/6c8bc701-6644-54f8-94af-291fdfbab41c/scratchpad/forecasting_project/outputs


### Lectura del EDA

- La serie **no es estacionaria** en niveles (prueba ADF, p > 0.05) — tiene una
  tendencia de crecimiento clara.
- Al diferenciar con periodo semanal (d=7) **sí se vuelve estacionaria**
  (p < 0.001), confirmando que la estacionalidad dominante es semanal.
- La descomposición muestra un patrón semanal consistente: lunes/martes son los
  días de mayor volumen, domingo el de menor — comportamiento típico de una
  operación de servicio al cliente.


## 3. Modelo de pronóstico (SARIMAX)

Se usa SARIMAX con estacionalidad semanal (periodo=7) y términos de Fourier como variables exógenas para capturar la estacionalidad anual (un ciclo demasiado largo para el componente estacional nativo de SARIMA).

In [3]:
"""
03_modelo_forecasting.py

Modelo de pronostico de volumen de contactos con SARIMAX (statsmodels):
- Estacionalidad semanal modelada con el componente estacional SARIMA (periodo=7)
- Estacionalidad anual modelada con terminos de Fourier (seno/coseno del dia del
  anio) como variables exogenas, tecnica estandar cuando una serie tiene mas de
  un ciclo estacional y el periodo anual (365) es demasiado largo para el
  componente estacional nativo de SARIMA.
- Validacion con holdout de los ultimos 60 dias (fuera de muestra).
- Metricas: MAPE y RMSE.
- Pronostico extendido 90 dias hacia adelante para alimentar la app de escenarios.

NOTA TECNICA (diagnostico real durante el desarrollo):
Una primera version con doble diferenciacion (d=1 regular + D=1 estacional)
producia un MAPE de ~174% en el holdout: el pronostico "explotaba" exponencialmente
en horizontes largos (60 dias), un efecto clasico de sobre-diferenciar una serie
que ya tiene una tendencia predominantemente lineal. La solucion fue modelar la
tendencia de forma explicita con el parametro trend="t" de SARIMAX, dejando solo
la diferenciacion estacional (D=1, periodo=7) para la estacionalidad semanal
(d=0 en la parte regular). Este cambio bajo el MAPE de validacion a ~12%.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX

BASE = Path.cwd()
OUT = BASE / "outputs"
DATA = BASE / "data"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA / "volumen_contactos.csv", parse_dates=["fecha"])
df = df.set_index("fecha").asfreq("D")
serie = df["volumen_contactos"]


def fourier_terms(index, periodo=365.25, n_armonicos=2):
    """Terminos de Fourier (seno/coseno) para modelar estacionalidad anual como exogena."""
    dia_del_anio = index.dayofyear.values
    terms = {}
    for k in range(1, n_armonicos + 1):
        terms[f"sin_{k}"] = np.sin(2 * np.pi * k * dia_del_anio / periodo)
        terms[f"cos_{k}"] = np.cos(2 * np.pi * k * dia_del_anio / periodo)
    return pd.DataFrame(terms, index=index)


exog_completo = fourier_terms(serie.index)

# --- Split train / test (holdout de 60 dias) ---
HOLDOUT = 60
train_y, test_y = serie.iloc[:-HOLDOUT], serie.iloc[-HOLDOUT:]
train_x, test_x = exog_completo.iloc[:-HOLDOUT], exog_completo.iloc[-HOLDOUT:]

modelo = SARIMAX(
    train_y,
    exog=train_x,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 7),
    trend="t",
    enforce_stationarity=False,
    enforce_invertibility=False,
)
resultado = modelo.fit(disp=False, maxiter=200)
print(resultado.summary().tables[0])

# --- Validacion en holdout ---
pred = resultado.get_forecast(steps=HOLDOUT, exog=test_x)
pred_media = pred.predicted_mean
pred_ic = pred.conf_int(alpha=0.10)  # intervalo de confianza 90%

mape = float(np.mean(np.abs((test_y - pred_media) / test_y)) * 100)
rmse = float(np.sqrt(np.mean((test_y - pred_media) ** 2)))
print(f"\n=== Validacion (holdout {HOLDOUT} dias) ===")
print(f"MAPE: {mape:.2f}%")
print(f"RMSE: {rmse:.1f} contactos/dia")

# --- Grafico de validacion ---
fig, ax = plt.subplots(figsize=(12, 5))
train_y.iloc[-120:].plot(ax=ax, label="Historico (train)", color="#5B6577")
test_y.plot(ax=ax, label="Real (holdout)", color="#1F3864", linewidth=2)
pred_media.plot(ax=ax, label="Pronostico", color="#2E75B6", linewidth=2, linestyle="--")
ax.fill_between(pred_ic.index, pred_ic.iloc[:, 0], pred_ic.iloc[:, 1],
                 color="#2E75B6", alpha=0.15, label="Intervalo de confianza 90%")
ax.set_title(f"Validacion del modelo (MAPE={mape:.1f}%, RMSE={rmse:.0f})")
ax.set_ylabel("Contactos/dia")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "05_validacion_forecast.png", dpi=130)
plt.close()

# --- Reentrenar con TODA la serie y proyectar 90 dias hacia adelante ---
modelo_full = SARIMAX(
    serie,
    exog=exog_completo,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 7),
    trend="t",
    enforce_stationarity=False,
    enforce_invertibility=False,
)
resultado_full = modelo_full.fit(disp=False, maxiter=200)

FUTURO = 90
fechas_futuras = pd.date_range(serie.index[-1] + pd.Timedelta(days=1), periods=FUTURO, freq="D")
exog_futuro = fourier_terms(fechas_futuras)
pred_futuro = resultado_full.get_forecast(steps=FUTURO, exog=exog_futuro)
media_futuro = pred_futuro.predicted_mean
ic_futuro = pred_futuro.conf_int(alpha=0.10)

df_forecast = pd.DataFrame({
    "fecha": fechas_futuras,
    "volumen_pronosticado": media_futuro.values,
    "ic_inferior_90": ic_futuro.iloc[:, 0].values,
    "ic_superior_90": ic_futuro.iloc[:, 1].values,
})
df_forecast["volumen_pronosticado"] = df_forecast["volumen_pronosticado"].clip(lower=0)
df_forecast["ic_inferior_90"] = df_forecast["ic_inferior_90"].clip(lower=0)
df_forecast.to_csv(DATA / "forecast_90_dias.csv", index=False)

# Guardar tambien metricas de validacion para mostrarlas en la app / README
pd.DataFrame([{"metrica": "MAPE", "valor": round(mape, 2)},
              {"metrica": "RMSE", "valor": round(rmse, 1)},
              {"metrica": "dias_holdout", "valor": HOLDOUT}]).to_csv(
    DATA / "metricas_validacion.csv", index=False
)

# --- Grafico del pronostico futuro ---
fig, ax = plt.subplots(figsize=(12, 5))
serie.iloc[-150:].plot(ax=ax, label="Historico", color="#5B6577")
media_futuro.plot(ax=ax, label="Pronostico (90 dias)", color="#1F7A4D", linewidth=2)
ax.fill_between(fechas_futuras, ic_futuro.iloc[:, 0], ic_futuro.iloc[:, 1],
                 color="#1F7A4D", alpha=0.15, label="Intervalo de confianza 90%")
ax.set_title("Pronostico de volumen de contactos - proximos 90 dias")
ax.set_ylabel("Contactos/dia")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "06_pronostico_90_dias.png", dpi=130)
plt.close()

print(f"\nPronostico guardado en: {DATA / 'forecast_90_dias.csv'}")
print(f"Figuras guardadas en: {OUT}")


                                     SARIMAX Results                                     
Dep. Variable:                 volumen_contactos   No. Observations:                 1036
Model:             SARIMAX(1, 0, 1)x(1, 1, 1, 7)   Log Likelihood               -6986.996
Date:                           Wed, 23 Sep 2026   AIC                          13993.991
Time:                                   09:07:07   BIC                          14043.267
Sample:                               01-01-2023   HQIC                         14012.701
                                    - 11-01-2025                                         
Covariance Type:                             opg                                         

=== Validacion (holdout 60 dias) ===
MAPE: 12.14%
RMSE: 253.0 contactos/dia



Pronostico guardado en: /tmp/claude-0/-home-claude/6c8bc701-6644-54f8-94af-291fdfbab41c/scratchpad/forecasting_project/data/forecast_90_dias.csv
Figuras guardadas en: /tmp/claude-0/-home-claude/6c8bc701-6644-54f8-94af-291fdfbab41c/scratchpad/forecasting_project/outputs


### Resultado de la validación

El modelo se validó contra 60 días reales fuera de muestra (holdout), alcanzando
un **MAPE de ~12%** — un nivel de precisión razonable para un pronóstico diario
con doble estacionalidad.

**Nota técnica honesta:** la primera versión del modelo (con doble
diferenciación, d=1 y D=1) producía un MAPE de ~174%, porque el pronóstico
"explotaba" exponencialmente en el horizonte de 60 días — un efecto clásico de
sobre-diferenciar una serie con tendencia predominantemente lineal. La solución
fue modelar la tendencia explícitamente (`trend="t"`) y dejar solo la
diferenciación estacional. Este tipo de diagnóstico y ajuste es parte real del
trabajo de modelado de series de tiempo.


## 4. De pronóstico a dotación (FTE) — Modelo Erlang C

Se traduce el volumen pronosticado en personal requerido usando el modelo Erlang C (estándar de la industria de centros de contacto), con supuestos de AHT, horas de operación, Nivel de Servicio objetivo y shrinkage.

In [4]:
"""
wfm_utils.py

Funciones de planeacion de personal (Workforce Management) que traducen un
volumen pronosticado de contactos en la dotacion (FTE) necesaria, usando el
modelo Erlang C -- el estandar de la industria de centros de contacto para
calcular cuantos agentes se necesitan para alcanzar un Nivel de Servicio (NS)
objetivo (ej. "80% de las llamadas contestadas en <= 20 segundos").

Se usa tanto en 04_calculo_dotacion.py como en la app de Streamlit (05_app_streamlit.py),
para que el mismo calculo se pueda explorar de forma interactiva por escenario.
"""

import math


def erlang_c_prob_espera(agentes: int, intensidad_erlangs: float) -> float:
    """Probabilidad de que un contacto tenga que esperar (formula de Erlang C)."""
    if agentes <= intensidad_erlangs:
        return 1.0  # sistema inestable: mas trafico del que los agentes pueden atender
    suma = sum((intensidad_erlangs ** n) / math.factorial(n) for n in range(agentes))
    ultimo_termino = (intensidad_erlangs ** agentes) / math.factorial(agentes)
    factor_ocupacion = agentes / (agentes - intensidad_erlangs)
    numerador = ultimo_termino * factor_ocupacion
    return numerador / (suma + numerador)


def nivel_de_servicio(agentes: int, intensidad_erlangs: float, aht_seg: float,
                       objetivo_seg: float) -> float:
    """NS = probabilidad de que un contacto sea atendido dentro de `objetivo_seg`."""
    if agentes <= intensidad_erlangs:
        return 0.0
    p_espera = erlang_c_prob_espera(agentes, intensidad_erlangs)
    exponente = -(agentes - intensidad_erlangs) * (objetivo_seg / aht_seg)
    return 1 - p_espera * math.exp(exponente)


def agentes_requeridos_erlang_c(contactos_hora: float, aht_seg: float,
                                 ns_objetivo: float, tiempo_objetivo_seg: float,
                                 max_agentes: int = 400) -> int:
    """
    Numero minimo de agentes 'en base' (sin shrinkage) que cumple el NS objetivo,
    buscando el minimo m tal que nivel_de_servicio(m) >= ns_objetivo.
    """
    intensidad = (contactos_hora * aht_seg) / 3600.0  # trafico en Erlangs
    agentes = max(1, math.ceil(intensidad))
    while agentes < max_agentes:
        if nivel_de_servicio(agentes, intensidad, aht_seg, tiempo_objetivo_seg) >= ns_objetivo:
            return agentes
        agentes += 1
    return max_agentes


def dotacion_diaria(volumen_dia: float, horas_operacion: float, aht_seg: float,
                     ns_objetivo: float, tiempo_objetivo_seg: float,
                     shrinkage: float) -> dict:
    """
    A partir del volumen total del dia, calcula:
      - contactos_hora: volumen distribuido uniformemente en las horas de operacion
        (simplificacion razonable para un ejercicio de portafolio; en una operacion
        real esto se haria por intervalos de 30 min con su propio perfil de trafico).
      - agentes_base: agentes necesarios para el NS objetivo (Erlang C).
      - fte_con_shrinkage: agentes base ajustados por shrinkage (ausentismo, pausas,
        formacion, tiempo administrativo) para llegar a la dotacion real necesaria.
    """
    contactos_hora = volumen_dia / horas_operacion
    agentes_base = agentes_requeridos_erlang_c(
        contactos_hora, aht_seg, ns_objetivo, tiempo_objetivo_seg
    )
    fte_con_shrinkage = math.ceil(agentes_base / (1 - shrinkage))
    return {
        "contactos_hora": round(contactos_hora, 1),
        "agentes_base": agentes_base,
        "fte_con_shrinkage": fte_con_shrinkage,
    }


In [5]:
"""
04_calculo_dotacion.py

Traduce el pronostico de volumen (90 dias) en dotacion diaria requerida (FTE),
usando el modelo Erlang C de wfm_utils.py con supuestos base (caso "actual"):

    AHT (tiempo promedio de atencion): 360 segundos (6 minutos)
    Horas de operacion:                10 horas/dia
    Nivel de servicio objetivo:        80% de contactos atendidos
    Tiempo objetivo de respuesta:      20 segundos
    Shrinkage:                         32% (ausentismo, pausas, formacion, adm.)

Estos supuestos son ajustables interactivamente en la app de Streamlit
(05_app_streamlit.py) para simular escenarios.
"""

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from wfm_utils import dotacion_diaria

BASE = Path.cwd()
DATA = BASE / "data"
OUT = BASE / "outputs"

# --- Supuestos del caso base ---
AHT_SEG = 360
HORAS_OPERACION = 10
NS_OBJETIVO = 0.80
TIEMPO_OBJETIVO_SEG = 20
SHRINKAGE = 0.32

df = pd.read_csv(DATA / "forecast_90_dias.csv", parse_dates=["fecha"])

resultados = df["volumen_pronosticado"].apply(
    lambda v: dotacion_diaria(v, HORAS_OPERACION, AHT_SEG, NS_OBJETIVO, TIEMPO_OBJETIVO_SEG, SHRINKAGE)
)
df["contactos_hora"] = resultados.apply(lambda r: r["contactos_hora"])
df["agentes_base"] = resultados.apply(lambda r: r["agentes_base"])
df["fte_requerido"] = resultados.apply(lambda r: r["fte_con_shrinkage"])

df.to_csv(DATA / "dotacion_90_dias.csv", index=False)

print("=== Resumen de dotacion proyectada (proximos 90 dias) ===")
print(f"FTE promedio requerido: {df['fte_requerido'].mean():.1f}")
print(f"FTE minimo (dia mas bajo): {df['fte_requerido'].min()}")
print(f"FTE maximo (dia mas alto): {df['fte_requerido'].max()}")
print(f"\nSupuestos: AHT={AHT_SEG}s | Horas op.={HORAS_OPERACION}h | "
      f"NS objetivo={NS_OBJETIVO:.0%} en {TIEMPO_OBJETIVO_SEG}s | Shrinkage={SHRINKAGE:.0%}")

# --- Grafico de dotacion proyectada ---
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(df["fecha"], df["fte_requerido"], color="#2E75B6", alpha=0.75, label="FTE requerido")
ax1.set_ylabel("FTE requerido", color="#1F3864")
ax1.set_title("Dotacion proyectada (FTE) - proximos 90 dias")
ax2 = ax1.twinx()
ax2.plot(df["fecha"], df["volumen_pronosticado"], color="#B4720F", linewidth=1.5, label="Volumen pronosticado")
ax2.set_ylabel("Volumen pronosticado (contactos/dia)", color="#B4720F")
fig.tight_layout()
plt.savefig(OUT / "07_dotacion_proyectada.png", dpi=130)
plt.close()

print(f"\nGuardado en: {DATA / 'dotacion_90_dias.csv'}")
print(f"Figura: {OUT / '07_dotacion_proyectada.png'}")


=== Resumen de dotacion proyectada (proximos 90 dias) ===
FTE promedio requerido: 33.0
FTE minimo (dia mas bajo): 23
FTE maximo (dia mas alto): 39

Supuestos: AHT=360s | Horas op.=10h | NS objetivo=80% en 20s | Shrinkage=32%



Guardado en: /tmp/claude-0/-home-claude/6c8bc701-6644-54f8-94af-291fdfbab41c/scratchpad/forecasting_project/data/dotacion_90_dias.csv
Figura: /tmp/claude-0/-home-claude/6c8bc701-6644-54f8-94af-291fdfbab41c/scratchpad/forecasting_project/outputs/07_dotacion_proyectada.png


## 5. App interactiva de escenarios

El notebook muestra el flujo completo con supuestos fijos. Para explorar
escenarios de forma interactiva (crecimiento de volumen, AHT, NS objetivo,
shrinkage) y ver el impacto inmediato en la dotación proyectada, se construyó
una app en Streamlit: `05_app_streamlit.py`.

Para ejecutarla localmente:

```bash
streamlit run 05_app_streamlit.py
```

## Conclusiones

- El modelo de pronóstico captura correctamente la estacionalidad semanal y
  anual de la demanda, con un error de validación (~12% MAPE) razonable para
  planeación operativa.
- Traducir el pronóstico a dotación con Erlang C (en vez de una regla simple
  de proporcionalidad) refleja cómo se calcula la dotación real en operaciones
  de servicio al cliente, considerando el efecto no lineal del Nivel de
  Servicio objetivo.
- La app de escenarios permite a un equipo de planeación evaluar rápidamente
  el impacto de decisiones de negocio (una campaña, un cambio de NS objetivo,
  un ajuste de shrinkage) sin depender de recalcular todo manualmente.

**Repositorio y código completo:** github.com/Melchiah04
